In [9]:
using Pkg
Pkg.activate("../../..")
Pkg.instantiate()

  Activating project at `c:\Users\SurfacePro8\Documents\Studium\Master Thesis\Theoretical Model`


In [2]:
using DataFrames, JLD2
df = load_object("Measurements.jld2")

Row,Model,Extinction_Time,P_est,Replicate
,Symbol,Float64,Float64,Int64
1,pop_vol,360.0,0.372781,1
2,pop_vol,360.0,0.293103,2
3,pop_vol,360.0,0.319527,3
4,pop_vol,360.0,0.337143,4
5,pop_vol,360.0,0.364706,5
6,pop_vol,360.0,0.321839,6
7,pop_vol,360.0,0.323529,7
8,pop_vol,360.0,0.327485,8
9,pop_vol,360.0,0.331395,9


In [3]:
df.Extinction_Time .= df.Extinction_Time ./ 24.0

2120-element Vector{Float64}:
 15.0
 15.0
 15.0
 15.0
 15.0
 15.0
 15.0
 15.0
 15.0
 15.0
  ⋮
 15.0
 15.0
 15.0
 15.0
 15.0
 15.0
 15.0
 15.0
 15.0

Load Survival Time Data from Large Pop Evo v2 Experiment 

In [5]:
@load "Experimental Data" sub_df_mono_dp sub_df_cc_dp sub_df_co_poo

3-element Vector{Symbol}:
 :sub_df_mono_dp
 :sub_df_cc_dp
 :sub_df_co_poo

In [6]:
function plot_survival!(ax, df, model_syms, labels; color)
    sub = filter(row -> row.Model in model_syms, df)

    model_to_num = Dict(s => float(i) for (i, s) in enumerate(model_syms))

    violin!(
        ax,
        [model_to_num[s] for s in sub.Model],                  # categorical grouping
        sub.Extinction_Time,
        bandwidth = 0.2,
        color = color,
        scale = :area, 
        show_median = true
    )

    ax.xticks = (1:length(labels), labels)
end

function plot_pest!(ax, df, model_syms, labels; color)
    sub = filter(row -> row.Model in model_syms, df)

    model_to_num = Dict(s => float(i) for (i, s) in enumerate(model_syms))

    boxplot!(
        ax,
        [model_to_num[s] for s in sub.Model],
        sub.P_est,
        width = 0.4,
        whiskerwidth = 0.15,
        color = color, 
        show_outliers = false
    )

    ax.xticks = (1:length(labels), labels)
    ax.xticklabelsize = 10
end



plot_pest! (generic function with 1 method)

In [ ]:
using CairoMakie

model_names = ["Mono", "Establishment", "Demographic"]

fig = Figure()

grid = fig[1,1] = GridLayout()

model_names = ["Mono", "Establishment Model", "Demographic Model"]

model_groups = [
    [:mono],
    [:int_co, :int_vol, :int_pool],
    [:pop_co, :pop_vol, :pop_pool]
]

plot_labels = [
    ["Standard"],
    ["Standard", "Large", "Meta"],
    ["Standard", "Large", "Meta"]
]

main_axis = Axis(fig[1, 1], 
title = "Pooling Differentially Affects Survival and Establishment \n Across Interaction Mechanisms", 
titlesize = 13
)

hidedecorations!(main_axis)
hidespines!(main_axis)

surv_axes = [Axis(grid[2,i], ylabel = i==1 ? "Survival" : "", 
limits = (nothing, nothing, 5, 16.5),
yticks = [5, 10, 15]) for i in 1:3]
linkyaxes!(surv_axes...)

for i in 1:3
    Box(grid[1,i], color = :gray97)
    Label(grid[1,i], model_names[i], tellheight = 1, tellwidth = false, fontsize = 14)
end

colors = Makie.wong_colors()

for i in 1:3
    plot_survival!(
        surv_axes[i],
        df,
        model_groups[i],
        plot_labels[i],
        color = colors[1]
    )
end

pest_axes = [Axis(grid[3,i], ylabel = i==1 ? L"p_{\mathrm{est}}" : "",
limits = (nothing, nothing, 0.0, 0.6),
yticks = [0, 0.25, 0.5]) for i in 1:3]

for a in 1:3
    surv_axes[a].xticklabelsvisible = false
    surv_axes[a].xticksvisible = false
    surv_axes[a].ygridstyle = :dash
    pest_axes[a].ygridstyle = :dash
    surv_axes[a].xgridvisible = false
    pest_axes[a].xgridvisible = false
end

for a in 2:3
    hidedecorations!(surv_axes[a], grid = false)
    pest_axes[a].yticklabelsvisible = false
    pest_axes[a].yticksvisible = false
end

linkyaxes!(pest_axes...)

for a in 2:3
    xlims!(surv_axes[a], 0.5, 3.5)
    xlims!(pest_axes[a], 0.5, 3.5)
end
xlims!(surv_axes[1], 0.5, 1.5)

xlims!(pest_axes[1], 0.5, 1.5)

for i in 1:3
    plot_pest!(
        pest_axes[i],
        df,
        model_groups[i],
        plot_labels[i],
        color = colors[3]
    )
end

offset = 0.0

# Survival Time Density Plots of single-wells 
dens_ax = Axis(grid[4, 1:3],
xlabel = "Survival", 
ylabel = "Density",
ygridvisible = false, 
xgridvisible = false)

CairoMakie.density!(dens_ax, df[df.Model .== :pop_co, :].Extinction_Time, label = "Demographic Model", 
color = (colors[6], 0.4), transparency = true, bandwidth = 0.2, strokewidth = 0.5, strokecolor = colors[6])
CairoMakie.density!(dens_ax, df[df.Model .== :int_co, :].Extinction_Time, label = "Establishment Model", 
color = (colors[3], 0.4), transparency = true, bandwidth = 0.2, strokewidth = 0.5, strokecolor = colors[3])
CairoMakie.density!(dens_ax, sub_df_cc_dp.survived_time .+ offset, label = "Experimental Data", 
color = (colors[2], 0.4), transparency = true, bandwidth = 0.2, strokewidth = 0.5, strokecolor = colors[2])


axislegend(dens_ax, patchsize = (10, 10), labelsize = 8, framevisible = false)

rowgap!(grid, 15)
colgap!(grid, 5)

rowgap!(grid, 1, 0)
rowgap!(grid, 2, 10)

colsize!(grid, 1, Relative(1/7))
colsize!(grid, 2, Relative(3/7))
colsize!(grid, 3, Relative(3/7))

for (label, layout) in zip(["a", "b", "c"], [grid[2, :], grid[3, :], grid[4, :]])
    Label(layout[1, 1, Right()], label,
    fontsize = 20, 
    font = :bold, 
    padding = (5, 5, 5, 0))
end

save("Extinction Times + P_est Boxplot.png", fig)

In [10]:
filter(row -> (row.Model .== :int_co) .& (6.9 .< row.Extinction_Time .< 7.1), df)

Row,Model,Extinction_Time,P_est,Replicate
,Symbol,Float64,Float64,Int64
1,int_co,7.0,0.04,1
2,int_co,7.0,0.0392157,4
3,int_co,7.0,0.0285714,6
4,int_co,7.0,0.0480769,7
5,int_co,7.0,0.0594059,10
6,int_co,7.0,0.039604,11
7,int_co,6.99653,0.0606061,12
8,int_co,7.0,0.0480769,13
9,int_co,6.90347,0.0384615,14
